# Banana Ripeness — DINOv3 Embeddings

Download the Banana Ripeness Classification dataset, stage it locally, and extract DINOv3 embeddings.

In [ ]:
import shutil
from pathlib import Path

import kagglehub as kh
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

In [ ]:
# Config
DATASET_SLUG = "shahriar26s/banana-ripeness-classification-dataset"
DATASET_DIR_NAME = "Banana Ripeness Classification Dataset"
WORK_DIR = Path("/tmp/bananafp")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
MODEL_ID = "facebook/dinov3-vits16plus-pretrain-lvd1689m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Download dataset (cached by kagglehub)
cache_path = Path(kh.dataset_download(DATASET_SLUG))
print(f"Dataset cached at: {cache_path}")

In [ ]:
# Stage to work dir and collect image paths
src = cache_path / DATASET_DIR_NAME if (cache_path / DATASET_DIR_NAME).exists() else cache_path
dataset = WORK_DIR / DATASET_DIR_NAME

if not dataset.exists():
    shutil.copytree(src, dataset)

image_paths = sorted(p for p in dataset.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
print(f"{len(image_paths)} images in {dataset}")
print(image_paths[:5])

In [ ]:
# Extract embeddings with DINOv3 (transformers)
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(DEVICE)

@torch.no_grad()
def embed(pil_images, batch_size: int = 32):  # list[PIL.Image] -> (N, 384)
    """Embed a list of PIL images, batched to avoid OOM."""
    feats = []
    for i in range(0, len(pil_images), batch_size):
        batch = pil_images[i : i + batch_size]
        inputs = processor(images=batch, return_tensors="pt").to(DEVICE)
        feats.append(model(**inputs).pooler_output.cpu())
    return torch.cat(feats).numpy() if feats else None

In [ ]:
# Example: embed one image
# pil = Image.open(image_paths[0]).convert("RGB")
# vec = embed([pil])
# print(vec.shape)